# Phase 2 — Standard QAT (INT8 & INT4)

**Goal:** measure what one full pass of QAT with fake-quantization-from-step-1 achieves. This is the standard baseline that Scheduled QAT (notebook 04) has to beat to justify the schedule.

**How it works:** fake-quantization nodes are injected from epoch 0. The forward pass quantizes-then-dequantizes weights at the target precision so the model sees the precision loss; gradients flow through the rounding via the Straight-Through Estimator.

**Inputs:**
- `/kaggle/input/sqat-baseline/results/baseline/fp32_logits.pt`

**Outputs:**
- `models/checkpoints/standard_qat_int{4,8}/final.pt`
- `results/standard_qat_int{4,8}/training_results.json`

**Estimated time on T4:** ~3.5 h total (≈1.7 h per bit-width with the Kaggle config below).

## 1. Setup

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

WORKING_DIR  = "/kaggle/working"
REPO_DIR     = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL   = "https://github.com/JpCurada/scheduled-qat-for-slm.git"
REPO_DATASET = "/kaggle/input/sqat-repo"
BASELINE_DIR = "/kaggle/input/sqat-baseline/results/baseline"

if not os.path.exists(REPO_DIR):
    if os.path.exists(REPO_DATASET):
        shutil.copytree(REPO_DATASET, REPO_DIR)
    else:
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

FP32_LOGITS = Path(BASELINE_DIR) / "fp32_logits.pt"
assert FP32_LOGITS.exists(), f"FP32 logits not found — add sqat-baseline as input"
print(f"FP32 logits: {FP32_LOGITS}  ({FP32_LOGITS.stat().st_size / 1e9:.2f} GB)")

In [ ]:
!pip install -q -e ".[viz]"

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Kaggle config overrides

The published configs assume `seq_length=2048`, `epochs=3`, `batch_size=8`. That doesn't fit Kaggle's 12 h session limit on a T4 — at the published settings, INT4 alone takes ~15 h. We patch:

| Field | Original | Kaggle |
|---|---|---|
| `data.seq_length` | 2048 | 512 |
| `training.epochs` | 3 | 1 |
| `training.batch_size` | 8 | 4 |
| `training.gradient_accumulation_steps` | 4 | 8 (effective batch unchanged at 32) |
| `training.warmup_steps` | 500 | 100 |
| `logging.save_every_steps` | 5000 | 999999 (only final checkpoint) |
| `logging.eval_every_steps` | 2500 | 500 |

Effective batch size and learning rate are preserved so results are still comparable to the full-spec runs.

In [ ]:
import yaml

MODEL_CACHE   = f"{WORKING_DIR}/models/base"
KAGGLE_CFG    = Path(REPO_DIR) / "configs_kaggle" / "standard_qat"
KAGGLE_CFG.mkdir(parents=True, exist_ok=True)

def patch_qat_config(src: str, dst: Path) -> Path:
    with open(src) as f:
        cfg = yaml.safe_load(f)
    cfg["model"]["cache_dir"] = MODEL_CACHE
    cfg["data"]["seq_length"] = 512
    cfg["training"]["epochs"] = 1
    cfg["training"]["batch_size"] = 4
    cfg["training"]["gradient_accumulation_steps"] = 8
    cfg["training"]["warmup_steps"] = 100
    cfg["logging"]["save_every_steps"] = 999999
    cfg["logging"]["eval_every_steps"] = 500
    cfg["logging"]["log_dir"] = f"{WORKING_DIR}/results/logs/{dst.stem}/"
    cfg["export"]["output_dir"] = f"{WORKING_DIR}/models/gguf/"
    with dst.open("w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return dst

qat8_cfg = patch_qat_config("configs/standard_qat/qat_int8.yaml", KAGGLE_CFG / "qat_int8.yaml")
qat4_cfg = patch_qat_config("configs/standard_qat/qat_int4.yaml", KAGGLE_CFG / "qat_int4.yaml")
print("INT8 config:", qat8_cfg)
print("INT4 config:", qat4_cfg)

## 3. Standard QAT — INT8

INT8 is the easier target — fake-quant noise is small enough that even one epoch should recover most FP32 quality. Look for KL divergence noticeably below PTQ INT8.

In [ ]:
import gc
from src.training.trainer import run_experiment

results_int8 = run_experiment(
    config_path=str(qat8_cfg),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nStandard QAT INT8 results:")
for k, v in results_int8.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 4. Standard QAT — INT4

INT4 is where QAT earns its keep. The fake-quant noise is large from step 1; expect higher loss and noticeable KL divergence vs the FP32 baseline. This is the headline number that Scheduled QAT will try to beat.

In [ ]:
results_int4 = run_experiment(
    config_path=str(qat4_cfg),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nStandard QAT INT4 results:")
for k, v in results_int4.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 5. Training loss curves

Read JSONL step-logs from both runs and overlay the loss / val-perplexity curves. Helps identify if the training plateaus, diverges, or still has headroom.

In [ ]:
import json
import matplotlib.pyplot as plt

def read_jsonl(path):
    if not Path(path).exists():
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

log8 = read_jsonl(f"{WORKING_DIR}/results/logs/qat_int8/training_steps.jsonl")
log4 = read_jsonl(f"{WORKING_DIR}/results/logs/qat_int4/training_steps.jsonl")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, log, title in [(axes[0], log8, "INT8"), (axes[1], log4, "INT4")]:
    if not log:
        ax.text(0.5, 0.5, "no log entries", ha="center", va="center")
        ax.set_title(f"Standard QAT {title}")
        continue
    steps   = [e["step"]    for e in log]
    losses  = [e["loss"]    for e in log]
    val_ppl = [e["val_ppl"] for e in log]
    ax.plot(steps, losses, label="train loss", color="tab:blue")
    ax2 = ax.twinx()
    ax2.plot(steps, val_ppl, label="val ppl", color="tab:orange", linestyle="--")
    ax.set_xlabel("step");  ax.set_ylabel("loss", color="tab:blue")
    ax2.set_ylabel("val ppl", color="tab:orange")
    ax.set_title(f"Standard QAT {title}")
fig.tight_layout()
plt.show()

## 6. Comparison table

In [ ]:
import pandas as pd

with open(Path(BASELINE_DIR) / "baseline_results.json") as f:
    fp32 = json.load(f)

rows = [
    {"method": "FP32",         "bits": 32, **{k: fp32.get(k)         for k in ("perplexity",)}, "kl_divergence": 0.0},
    {"method": "Standard QAT", "bits": 8,  "perplexity": results_int8.get("perplexity"), "kl_divergence": results_int8.get("kl_divergence")},
    {"method": "Standard QAT", "bits": 4,  "perplexity": results_int4.get("perplexity"), "kl_divergence": results_int4.get("kl_divergence")},
]
df = pd.DataFrame(rows)
df["ppl_delta_pct"] = ((df["perplexity"] / df["perplexity"].iloc[0]) - 1.0) * 100
df.round(4)

In [ ]:
summary_path = Path(WORKING_DIR) / "results" / "standard_qat_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
with summary_path.open("w") as f:
    json.dump({"int8": results_int8, "int4": results_int4, "fp32": fp32},
              f, indent=2, default=str)
print(f"Wrote {summary_path}")
!ls -lh {WORKING_DIR}/models/checkpoints/

## Next steps

Save outputs as a Kaggle dataset (e.g. `sqat-standard-qat`). Notebooks 06 (export) and 07 (analysis) will mount it.

Proceed to `04_scheduled_qat.ipynb` — the main contribution.